# Custosell — API Reference

**Version:** 1.0  
**Base URL:** `http://localhost:8000/api/v1`  
**Auth:** Bearer token via `Authorization: Bearer {token}` header  
**Content-Type:** `application/json`  
**Currency:** UGX (Ugandan Shillings, all monetary values in decimal)

---
## Conventions

### Authentication
- Public endpoints: no token required
- Protected endpoints: require `Authorization: Bearer {token}` — returns 401 if missing/expired

### Business Scoping
All scoped entities (categories, products, customers, sales, etc.) automatically filter by the authenticated user's `business_id`. Staff users only see their own business's data.

### Pagination
All list endpoints return Laravel pagination format:
```json
{
  "data": [...],
  "links": { "first": "...", "last": "...", "prev": null, "next": "..." },
  "meta": { "current_page": 1, "from": 1, "last_page": 1, "per_page": 15, "to": 10, "total": 10 }
}
```

### Error Response Format
```json
{
  "message": "Error description",
  "errors": { "field_name": ["Validation error message"] }
}
```

### Status Codes
| Code | Meaning |
|------|---------|
| 200 | Success (GET, PUT) |
| 201 | Created (POST) |
| 204 | No content (DELETE) |
| 401 | Unauthenticated (missing/expired token) |
| 403 | Forbidden (no permission) |
| 404 | Not found |
| 409 | Conflict (e.g. duplicate active shift) |
| 422 | Validation error |

### Enum Values
| Field | Values |
|-------|--------|
| `payment_method` | `cash`, `mobile_money`, `card`, `other` |
| `payment_status` | `paid`, `partially_refunded`, `refunded` |
| `stock_movement_type` | `purchase`, `sale`, `adjustment`, `return`, `initial` |
| `shift_status` | `active`, `completed` |
| `subscription_status` | `active`, `trialing`, `cancelled`, `expired` |
| `business_status` | `active`, `suspended` |

---
## 1. Authentication

### POST `/auth/register`
**Auth:** None  
**Description:** Create a new user account. Returns user data + API token.

**Request Body:**
| Field | Type | Required | Rules |
|-------|------|----------|-------|
| name | string | Yes | max:255 |
| email | string | Yes | valid email, unique, max:255 |
| password | string | Yes | min:6 |
| password_confirmation | string | Yes | must match password |
| phone | string | No | max:50 |

**Response (201):**
```json
{
  "user": {
    "data": {
      "id": 1,
      "business_id": null,
      "role_id": null,
      "name": "John Doe",
      "email": "john@example.com",
      "phone": "+256701234567",
      "is_active": true,
      "role": null,
      "created_at": "2026-06-02T12:00:00.000000Z",
      "updated_at": "2026-06-02T12:00:00.000000Z"
    }
  },
  "token": "1|abc123tokenstring..."
}
```
**Errors:** 422 — email already taken, password too short, confirmation mismatch

### POST `/auth/login`
**Auth:** None  
**Description:** Authenticate and receive API token.

**Request Body:**
| Field | Type | Required | Rules |
|-------|------|----------|-------|
| email | string | Yes | valid email |
| password | string | Yes | - |

**Response (200):**
```json
{
  "user": {
    "data": {
      "id": 1,
      "business_id": 1,
      "role_id": 1,
      "name": "John Doe",
      "email": "john@example.com",
      "phone": "+256701234567",
      "is_active": true,
      "role": { "data": { "id": 1, "name": "Admin", "slug": "admin", "permissions": { "sales.create": true, "sales.view": true } } },
      "created_at": "2026-06-02T12:00:00.000000Z",
      "updated_at": "2026-06-02T12:00:00.000000Z"
    }
  },
  "token": "1|abc123tokenstring..."
}
```
**Errors:** 401 — invalid credentials

### POST `/auth/logout`
**Auth:** Required  
**Description:** Revoke current API token.  
**Response (200):** `{ "message": "Logged out" }`  
**Errors:** 401 — no token

### GET `/auth/me`
**Auth:** Required  
**Description:** Get currently authenticated user with role.  
**Response (200):** User resource with nested role and permissions

---
## 2. Plans

### GET `/plans`
**Auth:** None  
**Description:** List all plans ordered by sort_order.

**Response (200) — Seeded plans:**
```json
{
  "data": [
    {
      "id": 1, "name": "Free", "slug": "free",
      "price_monthly": "0.00", "price_yearly": null,
      "features": { "expenses": false, "shift_tracking": false, "discounts": false, "refunds": false, "export_data": false },
      "limits": { "staff_users": 1, "products": 50, "monthly_sales": 100, "customers": 50, "categories": 5 },
      "sort_order": 1
    },
    {
      "id": 2, "name": "Pro", "slug": "pro",
      "price_monthly": "30000.00", "price_yearly": "300000.00",
      "features": { "expenses": true, "shift_tracking": true, "discounts": true, "refunds": true, "export_data": false },
      "limits": { "staff_users": 5, "products": 1000, "monthly_sales": null, "customers": 1000, "categories": 30 },
      "sort_order": 2
    },
    {
      "id": 3, "name": "Premium", "slug": "premium",
      "price_monthly": "100000.00",
      "features": { "expenses": true, "shift_tracking": true, "discounts": true, "refunds": true, "export_data": true },
      "limits": { "staff_users": null, "products": null, "monthly_sales": null, "customers": null, "categories": null },
      "sort_order": 3
    }
  ]
}
```

**Frontend notes:** `null` limit = unlimited. Use `features` to show/hide UI. Use `limits` to enforce max counts.

### POST `/plans` — Create plan  
### GET `/plans/{id}` — Get single plan  
### PUT `/plans/{id}` — Update plan  
### DELETE `/plans/{id}` — Delete plan (204)

---
## 3. Businesses

### POST `/businesses/register`
**Auth:** None  
**Description:** Register a new business with its owner user. Creates both in one transaction.

**Request Body:**
| Field | Type | Required | Rules |
|-------|------|----------|-------|
| name | string | Yes | Business name, max:255 |
| slug | string | No | Auto-generated from name if omitted |
| email | string | No | |
| phone | string | No | |
| address | string | No | |
| currency | string | No | Default: "UGX" |

**Response (201):**
```json
{
  "data": {
    "id": 1,
    "owner_id": 1,
    "name": "My Shop",
    "slug": "my-shop",
    "email": "shop@example.com",
    "phone": "+256701234567",
    "address": "123 Kampala Road",
    "currency": "UGX",
    "receipt_footer": null,
    "logo_path": null,
    "status": "active",
    "trial_ends_at": null,
    "subscription": null,
    "created_at": "2026-06-02T12:00:00.000000Z",
    "updated_at": "2026-06-02T12:00:00.000000Z"
  }
}
```

**Frontend:** After registration, user is created as business owner. Use `/auth/login` to get a token.

### GET `/businesses/mine` — Owner's business  
### GET `/businesses/{id}` — Get by ID  
### PUT `/businesses/{id}` — Update  
### GET `/businesses/settings` — Get settings  
### PUT `/businesses/settings` — Update settings

---
## 4. Roles

### GET `/roles`
**Auth:** Required  
**Response (200):**
```json
{
  "data": [
    {
      "id": 1, "name": "Admin", "slug": "admin",
      "permissions": { "sales.create": true, "sales.view": true, "sales.refund": true },
      "is_default": true
    },
    {
      "id": 2, "name": "Staff", "slug": "staff",
      "permissions": { "sales.create": true, "sales.view": true, "sales.refund": false },
      "is_default": true
    }
  ]
}
```

**Standard CRUD:** POST, GET, PUT, DELETE `/roles/{id}`

**Permission flags for frontend UI gating:**
```javascript
if (user.role.permissions['sales.refund']) showRefundButton()
if (user.role.permissions['expenses.view']) showExpensesMenuItem()
if (user.role.permissions['users.create']) showAddStaffButton()
```

**All permission keys:** `sales.create`, `sales.view`, `sales.refund`, `sales.discount`, `sales.delete`, `inventory.view`, `inventory.create`, `inventory.edit`, `inventory.delete`, `customers.view`, `customers.create`, `customers.edit`, `expenses.view`, `expenses.create`, `expenses.edit`, `expenses.delete`, `users.view`, `users.create`, `users.edit`, `users.delete`, `reports.view`, `settings.view`, `settings.edit`

---
## 5. Users (Staff Management)

### GET `/users`
**Auth:** Required  
**Response (200):** List of staff users scoped to business:
```json
{
  "data": [{
    "id": 2, "business_id": 1, "role_id": 2,
    "name": "Jane Staff", "email": "jane@example.com", "phone": "+256701234568",
    "is_active": true,
    "role": { "data": { "id": 2, "name": "Staff", "slug": "staff" } },
    "created_at": "2026-06-02T12:00:00.000000Z",
    "updated_at": "2026-06-02T12:00:00.000000Z"
  }]
}
```

### POST `/users` — Create staff  
| Field | Type | Required | Rules |
|-------|------|----------|-------|
| name | string | Yes | max:255 |
| email | string | Yes | unique |
| password | string | Yes | min:6 |
| password_confirmation | string | Yes | |
| phone | string | No | |
| role_id | integer | No | |

**Frontend:** `created_by` and `business_id` auto-set from authenticated user.

### GET `/users/{id}` — Get staff  
### DELETE `/users/{id}` — Soft delete (204)

---
## 6. Categories

### `GET/POST /categories`  
### `GET/PUT/DELETE /categories/{id}`

**Auth:** Required  

**Response (200):**
```json
{
  "data": [{
    "id": 1, "business_id": 1, "name": "Beverages",
    "description": "Drinks and refreshments", "sort_order": 0,
    "created_at": "2026-06-02T12:00:00.000000Z",
    "updated_at": "2026-06-02T12:00:00.000000Z"
  }]
}
```

**POST body:** `name` (required, unique per business), `description` (no), `sort_order` (no, default 0)

---
## 7. Products

### GET `/products`
**Auth:** Required  
**Response (200):**
```json
{
  "data": [{
    "id": 1, "business_id": 1, "category_id": 1,
    "name": "Coca Cola 500ml", "sku": "COKE-500",
    "unit_price": "3000.00", "cost_price": "2000.00",
    "stock_quantity": 50, "low_stock_threshold": 5,
    "tax_percentage": "0.00", "is_active": true,
    "category": { "data": { "id": 1, "name": "Beverages" } },
    "created_at": "2026-06-02T12:00:00.000000Z",
    "updated_at": "2026-06-02T12:00:00.000000Z"
  }]
}
```

### POST `/products`  
| Field | Type | Required | Notes |
|-------|------|----------|-------|
| name | string | Yes | |
| category_id | int | No | |
| unit_price | number | Yes | Selling price in UGX |
| cost_price | number | No | For profit calc |
| stock_quantity | int | No | Default 0 |
| low_stock_threshold | int | No | Default 5 |
| sku | string | No | Unique per business |

### GET `/products/low-stock` — Products where `stock_quantity <= low_stock_threshold`  
### GET `/products/{id}/stock-movements` — Inventory ledger for one product  
### PUT `/products/{id}` — Update  
### DELETE `/products/{id}` — Soft delete (204)

---
## 8. Customers

### GET `/customers`  
**Response (200):**
```json
{
  "data": [{
    "id": 1, "name": "Alice Mukisa", "phone": "+256701234569",
    "email": "alice@example.com", "total_purchases": "15000.00",
    "last_purchase_at": "2026-06-02T12:00:00.000000Z",
    "created_at": "2026-06-02T12:00:00.000000Z"
  }]
}
```

### POST `/customers`  
| Field | Type | Required | Rules |
|-------|------|----------|-------|
| name | string | Yes | |
| phone | string | Yes | Unique per business |
| email | string | No | |

### GET `/customers/{id}/purchases` — Purchase history (returns paginated Sales)  
### PUT/DELETE `/customers/{id}`

---
## 9. Shifts

### POST `/shifts/clock-in`
**Auth:** Required  
**Description:** Start a shift. One active shift at a time.

**Response (201):**
```json
{
  "data": {
    "id": 1, "business_id": 1, "user_id": 1,
    "clock_in": "2026-06-02T08:00:00.000000Z", "clock_out": null,
    "total_sales": "0.00", "total_cash": "0.00",
    "total_mobile_money": "0.00", "total_card": "0.00",
    "status": "active",
    "created_at": "2026-06-02T08:00:00.000000Z"
  }
}
```

**Errors:** 409 — already have an active shift

### POST `/shifts/{id}/clock-out` — End shift (200)  
### GET `/shifts/active` — Current active shift  
### GET `/shifts` — List history (query: `?status=active|completed`, `?date=2026-06-02`)

---
## 10. Sales

### GET `/sales`
**Auth:** Required  
**Query:** `?date=2026-06-02`, `?customer_id=1`, `?payment_method=cash`  

**Response (200) — includes items:**
```json
{
  "data": [{
    "id": 1, "receipt_number": "my-shop-0001",
    "subtotal": "3000.00", "tax_total": "0.00", "discount_amount": "0.00",
    "total_amount": "3000.00", "payment_method": "cash",
    "payment_status": "paid",
    "sale_date": "2026-06-02T12:00:00.000000Z",
    "items": [{
      "id": 1, "product_name": "Coca Cola 500ml", "quantity": 1,
      "unit_price": "3000.00", "subtotal": "3000.00",
      "refunded_quantity": 0, "refunded_amount": "0.00"
    }]
  }]
}
```

### POST `/sales` — POS Checkout
**Request Body:**
| Field | Type | Required | Notes |
|-------|------|----------|-------|
| payment_method | string | Yes | `cash`, `mobile_money`, `card`, `other` |
| customer_id | int | No | |
| shift_id | int | No | |
| items | array | Yes | Array of `{product_id, quantity, unit_price}` |

**Side effects (all automatic):**
1. Sale items created with product name/price snapshots
2. Product `stock_quantity` decreased
3. StockMovement created with type `sale` + before/after
4. Receipt number auto-generated

### GET `/sales/daily` — Daily sales with aggregates in meta  
### GET `/sales/{id}` — Single sale with items  
### POST `/sales/{id}/refund` — Process refund (requires `sales.refund` permission)  
### DELETE `/sales/{id}` — Soft delete (204)

---
## 11. Sale Items

### GET `/sale-items?sale_id={id}`  
### POST `/sale-items`  
### PUT/DELETE `/sale-items/{id}`

**Auth:** Required  

**Response:**
```json
{
  "data": [{
    "id": 1, "product_id": 1,
    "product_name": "Coca Cola 500ml", "product_price": "3000.00",
    "quantity": 2, "unit_price": "3000.00", "subtotal": "6000.00",
    "refunded_quantity": 0, "refunded_amount": "0.00"
  }]
}
```

**Frontend:** `product_name` and `product_price` are snapshots frozen at sale time — they don't change if the product is later edited.

---
## 12. Stock Movements

### GET `/stock-movements?product_id=1&type=purchase`
**Auth:** Required  

**Response (200):**
```json
{
  "data": [{
    "id": 1, "product_id": 1, "type": "purchase",
    "quantity_change": 100, "stock_before": 50, "stock_after": 150,
    "reference": "PO-001", "notes": "Restock from supplier",
    "created_by": 1
  }]
}
```

**Frontend:** This is the audit trail. Every stock change recorded with before/after snapshots.

### POST `/stock-movements` — Manual adjustment
| Field | Type | Required | Notes |
|-------|------|----------|-------|
| product_id | int | Yes | |
| type | string | Yes | `purchase`, `adjustment`, `return`, `initial` |
| quantity_change | int | Yes | + for additions, - for reductions |
| reference | string | No | e.g. PO number |
| notes | string | No | |

**Side effect:** Product `stock_quantity` updated automatically.

---
## 13. Subscriptions

### GET `/subscriptions`
**Auth:** Required  
**Description:** Get the current business's subscription with plan details.

**Response (200):**
```json
{
  "data": {
    "id": 1, "business_id": 1, "plan_id": 1, "status": "active",
    "starts_at": "2026-06-02T12:00:00.000000Z", "trial_ends_at": null,
    "plan": {
      "data": {
        "id": 1, "name": "Free", "slug": "free",
        "price_monthly": "0.00",
        "features": { "expenses": false, "shift_tracking": false },
        "limits": { "staff_users": 1, "products": 50 }
      }
    }
  }
}
```

**Frontend:** Load on app init. Use `plan.features` to show/hide features. Use `plan.limits` to enforce max counts.

### POST `/subscriptions` — Create  
### PUT `/subscriptions/upgrade` — Change plan (`{ "plan_id": 2 }`)  
### POST `/subscriptions/cancel` — Cancel (sets `cancelled_at`)

---
## 14. Expense Categories

Standard CRUD: `GET/POST /expense-categories`, `GET/PUT/DELETE /expense-categories/{id}`

**Response:**
```json
{
  "data": [{
    "id": 1, "business_id": 1, "name": "Rent",
    "description": "Shop rent payments", "sort_order": 0,
    "created_at": "2026-06-02T12:00:00.000000Z"
  }]
}
```

---
## 15. Expenses

### GET `/expenses?category_id=1&date=2026-06-02`  

**Response:**
```json
{
  "data": [{
    "id": 1, "expense_category_id": 1, "amount": "500000.00",
    "description": "Monthly shop rent", "reference": "REC-001",
    "expense_date": "2026-06-01T00:00:00.000000Z",
    "category": { "data": { "id": 1, "name": "Rent" } }
  }]
}
```

### POST `/expenses`  
| Field | Type | Required | Rules |
|-------|------|----------|-------|
| expense_category_id | int | No | |
| amount | number | Yes | > 0 |
| description | string | Yes | |
| expense_date | string | Yes | ISO datetime |

**Standard CRUD:** PUT/DELETE `/expenses/{id}`

**Frontend:** Expenses used to calculate Net Sales on Dashboard. Net = Revenue − Expenses.

---
## 16. Sync (Electron Desktop)

### POST `/sync/push`
**Auth:** Required  
**Description:** Push local changes to cloud.

**Request:**
```json
{
  "categories": [{ "id": 1, "name": "Beverages" }],
  "products": [{ "id": 1, "name": "Coke", "unit_price": 3000, "stock_quantity": 50 }],
  "customers": [],
  "expenses": [],
  "sales": [],
  "sale_items": [],
  "stock_movements": []
}
```

**Response (200):**
```json
{
  "imported": { "categories": 5, "products": 10, "customers": 3, "expenses": 0, "sales": 8, "sale_items": 15, "stock_movements": 8 },
  "synced_at": "2026-06-02T12:00:00.000000Z"
}
```

### GET `/sync/pull?since=2026-06-01T00:00:00Z`
**Auth:** Required  
**Description:** Pull delta changes since timestamp.

**Response:** Returns `categories`, `products`, `customers`, `expense_categories`, `expenses`, `roles`, `shifts`, `sales`, `sale_items`, `stock_movements`, `users`, `synced_at`

### GET `/sync/full`
**Auth:** Required  
**Description:** Full data dump for first-time setup (seeding local SQLite).

**Frontend:** Call ONCE on first login to populate local SQLite. Then use `/sync/pull?since=` for delta updates.

---
## Quick Reference: Auth Flow

```javascript
// Login → store token
const login = async (email, password) => {
  const res = await fetch('/api/v1/auth/login', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ email, password })
  });
  const data = await res.json();
  localStorage.setItem('token', data.token);
  return data.user;
};

// Authenticated fetch helper
const api = (path, options = {}) => {
  const token = localStorage.getItem('token');
  return fetch(`/api/v1${path}`, {
    ...options,
    headers: {
      'Content-Type': 'application/json',
      'Authorization': `Bearer ${token}`,
      ...options.headers,
    },
  });
};

// Usage
const products = await api('/products').then(r => r.json());
const sale = await api('/sales', {
  method: 'POST',
  body: JSON.stringify({
    payment_method: 'cash',
    items: [{ product_id: 1, quantity: 2, unit_price: 3000 }]
  }),
}).then(r => r.json());
```

---
## Quick Reference: All Endpoints Summary

| Module | Method | Endpoint | Auth |
|--------|--------|----------|------|
| Auth | POST | `/auth/register` | No |
| Auth | POST | `/auth/login` | No |
| Auth | POST | `/auth/logout` | Yes |
| Auth | GET | `/auth/me` | Yes |
| Plans | GET | `/plans` | No |
| Plans | POST | `/plans` | Yes |
| Plans | GET | `/plans/{id}` | No |
| Plans | PUT | `/plans/{id}` | Yes |
| Plans | DELETE | `/plans/{id}` | Yes |
| Business | POST | `/businesses/register` | No |
| Business | GET | `/businesses/mine` | Yes |
| Business | GET | `/businesses/{id}` | Yes |
| Business | PUT | `/businesses/{id}` | Yes |
| Business | GET | `/businesses/settings` | Yes |
| Business | PUT | `/businesses/settings` | Yes |
| Roles | CRUD | `/roles` | Yes |
| Users | GET/POST | `/users` | Yes |
| Users | GET/DELETE | `/users/{id}` | Yes |
| Categories | CRUD | `/categories` | Yes |
| Products | GET/POST | `/products` | Yes |
| Products | GET | `/products/low-stock` | Yes |
| Products | GET | `/products/{id}/stock-movements` | Yes |
| Products | PUT/DELETE | `/products/{id}` | Yes |
| Customers | CRUD | `/customers` | Yes |
| Customers | GET | `/customers/{id}/purchases` | Yes |
| Shifts | POST | `/shifts/clock-in` | Yes |
| Shifts | POST | `/shifts/{id}/clock-out` | Yes |
| Shifts | GET | `/shifts/active` | Yes |
| Shifts | GET | `/shifts` | Yes |
| Sales | GET/POST | `/sales` | Yes |
| Sales | GET | `/sales/daily` | Yes |
| Sales | GET | `/sales/{id}` | Yes |
| Sales | POST | `/sales/{id}/refund` | Yes |
| Sales | DELETE | `/sales/{id}` | Yes |
| Sale Items | CRUD | `/sale-items` | Yes |
| Stock Movements | GET/POST | `/stock-movements` | Yes |
| Subscriptions | GET/POST | `/subscriptions` | Yes |
| Subscriptions | PUT | `/subscriptions/upgrade` | Yes |
| Subscriptions | POST | `/subscriptions/cancel` | Yes |
| Expense Categories | CRUD | `/expense-categories` | Yes |
| Expenses | CRUD | `/expenses` | Yes |
| Sync | POST | `/sync/push` | Yes |
| Sync | GET | `/sync/pull` | Yes |
| Sync | GET | `/sync/full` | Yes |